# Olist BI Project — v0.2.0 Exploratory Data Analysis

**Objective:** move from validated processed data to business-oriented exploratory analysis.

### Analytical principle
Each analysis keeps an explicit grain. We do **not** build one giant dataframe.

> v0.1.0 proves the data is usable.  
> v0.2.0 proves we can extract a business story from the data.

**Baseline:** `v0.1.0` processed outputs.


## 1. Setup & Data Loading

Load only the processed datasets tracked by v0.1.0.

**Deferred:** `geolocation_clean.csv` is intentionally excluded from v0.2.0 core analysis.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

ROOT = Path("..")
PROCESSED = ROOT / "data" / "processed"

DATASETS = {
    "orders": "orders_clean.csv",
    "order_items": "order_items_clean.csv",
    "order_payments": "order_payments_clean.csv",
    "order_reviews": "order_reviews_clean.csv",
    "customers": "customers_clean.csv",
    "sellers": "sellers_clean.csv",
    "products": "products_clean.csv",
    "category_translation": "category_translation_clean.csv",
}

dfs = {
    name: pd.read_csv(PROCESSED / filename)
    for name, filename in DATASETS.items()
}

for name, df in dfs.items():
    print(f"{name:22} rows={len(df):>7,}  cols={len(df.columns):>2}")


### Baseline schema check

The notebook should fail early if an expected processed dataset is missing.


In [ ]:
assert set(dfs) == set(DATASETS)

for name, df in dfs.items():
    assert not df.empty, f"{name} is unexpectedly empty"

orders = dfs["orders"]
order_items = dfs["order_items"]
order_payments = dfs["order_payments"]
order_reviews = dfs["order_reviews"]
customers = dfs["customers"]
sellers = dfs["sellers"]
products = dfs["products"]
category_translation = dfs["category_translation"]

print("Processed dataset load check: PASS")


In [ ]:
schema_summary = pd.DataFrame({
    "table": list(dfs.keys()),
    "rows": [len(df) for df in dfs.values()],
    "columns": [len(df.columns) for df in dfs.values()],
})
schema_summary


## 2. Analytical Definitions

### Source grain

| Dataset | Grain |
|---|---|
| orders | 1 row = 1 order |
| order_items | 1 row = 1 order item |
| order_payments | 1 row = 1 payment installment |
| order_reviews | 1 row = 1 review |
| customers | 1 row = 1 customer record |
| sellers | 1 row = 1 seller |
| products | 1 row = 1 product |
| category_translation | 1 row = 1 category translation |

### Analytical grain

| Dataframe | Grain |
|---|---|
| order_metrics | 1 row = 1 order |
| customer_metrics | 1 row = 1 customer_unique_id |
| seller_metrics | 1 row = 1 seller_id |
| category_metrics | 1 row = 1 product category |

### GMV definition

For v0.2.0, **GMV is defined as**:

`SUM(order_items.price)`

`payment_value` is not used as the primary GMV definition.


## 3. Order EDA

Questions:
- How many orders are in the processed dataset?
- What is the order-status distribution?
- How does order volume change over time?
- What does the completion/delivery pattern look like?


In [ ]:
# Standardize order timestamps needed for temporal analysis.
order_datetime_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in order_datetime_cols:
    if col in orders.columns:
        orders[col] = pd.to_datetime(orders[col], errors="coerce")

print(orders[["order_id", "order_status"]].head())
print(f"Unique orders: {orders['order_id'].nunique():,}")


In [ ]:
order_status_distribution = (
    orders["order_status"]
    .value_counts(dropna=False)
    .rename_axis("order_status")
    .reset_index(name="order_count")
)

order_status_distribution["share_pct"] = (
    order_status_distribution["order_count"]
    / order_status_distribution["order_count"].sum()
    * 100
)

assert order_status_distribution["order_count"].sum() == len(orders)

order_status_distribution


In [ ]:
ax = order_status_distribution.plot(
    x="order_status",
    y="order_count",
    kind="bar",
    legend=False,
    figsize=(9, 5),
    title="01 — Order Status Distribution",
)
ax.set_xlabel("Order status")
ax.set_ylabel("Number of orders")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
monthly_orders = (
    orders.dropna(subset=["order_purchase_timestamp"])
    .assign(
        order_month=lambda df: df["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()
    )
    .groupby("order_month", as_index=False)
    .agg(order_count=("order_id", "nunique"))
)

assert monthly_orders["order_month"].is_monotonic_increasing
assert monthly_orders["order_count"].sum() == orders["order_id"].nunique()

monthly_orders.head()


In [ ]:
ax = monthly_orders.plot(
    x="order_month",
    y="order_count",
    figsize=(11, 5),
    legend=False,
    title="02 — Monthly Orders",
)
ax.set_xlabel("Purchase month")
ax.set_ylabel("Number of orders")
plt.tight_layout()
plt.show()


## 4. Sales / Revenue EDA

**GMV source:** `order_items.price`

We derive order-level metrics without merging every source table:

- `order_gmv` = sum of item prices per order
- `item_count` = number of order-item rows per order
- `average_order_value` = order GMV per order

Shipping fees and payment values are kept conceptually separate from this core GMV definition.


In [ ]:
order_item_metrics = (
    order_items.groupby("order_id", as_index=False)
    .agg(
        order_gmv=("price", "sum"),
        item_count=("order_item_id", "count"),
    )
)

assert order_item_metrics["order_id"].is_unique
assert order_item_metrics["order_gmv"].notna().all()
assert order_item_metrics["item_count"].gt(0).all()

order_metrics = orders[
    ["order_id", "order_status", "order_purchase_timestamp"]
].merge(
    order_item_metrics,
    on="order_id",
    how="left",
    validate="one_to_one",
)

order_metrics["order_gmv"] = order_metrics["order_gmv"].fillna(0)
order_metrics["item_count"] = order_metrics["item_count"].fillna(0).astype(int)
order_metrics["average_order_value"] = order_metrics["order_gmv"]

assert order_metrics["order_id"].is_unique
assert len(order_metrics) == orders["order_id"].nunique()

order_metrics.head()


In [ ]:
sales_summary = pd.Series({
    "total_orders": order_metrics["order_id"].nunique(),
    "total_gmv": order_metrics["order_gmv"].sum(),
    "average_order_value": order_metrics["average_order_value"].mean(),
    "average_items_per_order": order_metrics["item_count"].mean(),
})

sales_summary


In [ ]:
monthly_gmv = (
    order_metrics.dropna(subset=["order_purchase_timestamp"])
    .assign(
        order_month=lambda df: df["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()
    )
    .groupby("order_month", as_index=False)
    .agg(
        gmv=("order_gmv", "sum"),
        order_count=("order_id", "nunique"),
    )
)

assert monthly_gmv["order_month"].is_monotonic_increasing
assert monthly_gmv["order_count"].sum() == order_metrics["order_id"].nunique()

monthly_gmv.head()


In [ ]:
ax = monthly_gmv.plot(
    x="order_month",
    y="gmv",
    figsize=(11, 5),
    legend=False,
    title="03 — Monthly GMV",
)
ax.set_xlabel("Purchase month")
ax.set_ylabel("GMV")
plt.tight_layout()
plt.show()


In [ ]:
ax = order_metrics["order_gmv"].plot(
    kind="hist",
    bins=50,
    figsize=(10, 5),
    title="Order GMV Distribution",
)
ax.set_xlabel("GMV per order")
plt.tight_layout()
plt.show()


## 5. Product & Category EDA

**Target analytical grain:** 1 row = 1 product category.

Planned outputs:
- `category_gmv`
- `category_volume`
- category contribution

Only the joins required for category analysis will be performed.


## 6. Customer EDA

**Target analytical grain:** 1 row = 1 `customer_unique_id`.

Planned metrics:
- unique customers
- orders per customer
- repeat customers
- customer GMV
- customer AOV / behavior groups


## 7. Seller EDA

**Target analytical grain:** 1 row = 1 `seller_id`.

Planned metrics:
- seller count
- seller GMV
- seller order volume
- seller contribution
- top sellers across multiple metrics


## 8. Review EDA

**Primary metric:** `review_score`

Planned analysis:
- score distribution
- average score
- score over time
- review score vs. delivery performance

Missing review comments are **not** treated as missing reviews.


## 9. Delivery EDA

Derived metrics:
- `delivery_days`
- `delivery_delay_days`

Planned analysis:
- delivery-time distribution
- estimated vs. actual delivery
- delivery performance by category/state where useful

Temporal anomalies retained by v0.1.0 are not silently removed.


## 10. EDA Findings & Business Questions

### Findings

> To be populated after the remaining analytical sections are executed and validated.

### Business questions for the next iteration

- Which categories contribute the most GMV and item volume?
- How concentrated is customer revenue among repeat customers?
- Which sellers contribute materially across both GMV and order volume?
- How strongly is delivery performance associated with review scores?
- Which operational patterns deserve deeper analysis in v0.3.0?

### v0.2.0 discipline

No dashboard is produced here. The objective is to establish validated patterns and business questions before advanced analytics and dashboard development.
